# Ruta B: Ciencia de Datos
## ICD Financiero S.A. | Optimización de la Gestión de Quejas

**Objetivo:** Identificar combinaciones de entidad, producto y motivo que concentran mayor riesgo 
regulatorio, y construir un modelo que anticipe ese riesgo.

**Dataset:** 1.044.110 registros trimestrales | 2014–2026 | Superintendencia Financiera de Colombia

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Configuración global de visualización
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('muted')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

## Sección 0: Setup y Carga

In [ ]:
DATA_PATH = Path('Challenge_DataAI_Candidatos.csv')

df = pd.read_csv(
    DATA_PATH,
    sep=';',
    encoding='latin-1',
    dtype={'CODIGO_ENTIDAD': str}
)

# Strip de espacios en nombres de columna
df.columns = df.columns.str.strip()

# Verificación inmediata de carga
print(f"Shape: {df.shape}")
print(f"\nTipos de dato:\n{df.dtypes}")
print(f"\nPrimeras filas:")
df.head(3)

/tmp/ipykernel_22450/1821635981.py:3: DtypeWarning: Columns (0: TIPO_ENTIDAD) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


Shape: (1044109, 18)

Tipos de dato:
TIPO_ENTIDAD                            object
CODIGO_ENTIDAD                             str
NOMBRE_ENTIDAD                             str
FECHA_CORTE                                str
UNIDAD_CAPTURA                         float64
NOMBRE_UNIDAD_CAPTURA                      str
CODIGO_PRODUCTO                        float64
PRODUCTO                                   str
CODIGO_MOTIVO                          float64
MOTIVO                                     str
QUEJAS_PENDIENTES                      float64
QUEJAS_RECIBIDAS                       float64
QUEJAS_FINALIZADAS                     float64
QUEJAS_FINALIZADAS_N1                  float64
QUEJAS_FINALIZADAS_N2                  float64
QUEJAS_EN_TRAMITE                      float64
QUEJAS_FAVOR_CONSUM_ACEP_ENT           float64
QUEJAS_FAVOR_CONSUM_NOACEP_ENT,,,,,        str
dtype: object

Primeras filas:


,TIPO_ENTIDAD,CODIGO_ENTIDAD,NOMBRE_ENTIDAD,FECHA_CORTE,UNIDAD_CAPTURA,NOMBRE_UNIDAD_CAPTURA,CODIGO_PRODUCTO,PRODUCTO,CODIGO_MOTIVO,MOTIVO,QUEJAS_PENDIENTES,QUEJAS_RECIBIDAS,QUEJAS_FINALIZADAS,QUEJAS_FINALIZADAS_N1,QUEJAS_FINALIZADAS_N2,QUEJAS_EN_TRAMITE,QUEJAS_FAVOR_CONSUM_ACEP_ENT,"QUEJAS_FAVOR_CONSUM_NOACEP_ENT,,,,,"
0,14,11,Banco SevoraMERICANA S.A.,30/06/2023,2.00,DEFENSORES DEL CONSUMIDOR FINANCIERO,28.00,SEGURO DE SALUD,135.00,INCUMPLIMIENTO DE OBLIGACIONES EN PRESTACI??N ...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,"0,,,,,"
1,14,11,Banco SevoraMERICANA S.A.,31/03/2024,1.00,ENTIDAD VIGILADA,36.00,SEGURO DE RENTAS VOLUNTARIAS,20.00,"DEMORA O NO DEVOLUCI??N DE SALDOS, APORTES O P...",0.00,0.00,0.00,0.00,0.00,0.00,0.00,"0,,,,"
2,13,18,Banco OximiaMERICANA S.A.,30/09/2024,2.00,DEFENSORES DEL CONSUMIDOR FINANCIERO,7.00,SEGURO DE SUSTRACCI??N,13.00,DEMORA O NO ENTREGA DEL CONTRATO O DE LA P??LIZA,0.00,0.00,0.00,0.00,0.00,0.00,0.00,"0,,,,,"


In [3]:
df.columns = df.columns.str.strip().str.rstrip(',') #QUEJAS_FAVOR_CONSUM_NOACEP_ENT sin ','

In [4]:
df['QUEJAS_FAVOR_CONSUM_NOACEP_ENT'] = (
    df['QUEJAS_FAVOR_CONSUM_NOACEP_ENT']
    .str.rstrip(',')
    .str.strip()
    .replace('', np.nan)
    .astype(float)
)

### Constantes del proyecto:

In [5]:
# Columnas agrupadas por rol analítico
COLS_ID        = ['TIPO_ENTIDAD', 'CODIGO_ENTIDAD', 'NOMBRE_ENTIDAD']
COLS_TIEMPO    = ['FECHA_CORTE']
COLS_PRODUCTO  = ['CODIGO_PRODUCTO', 'PRODUCTO']
COLS_MOTIVO    = ['CODIGO_MOTIVO', 'MOTIVO']
COLS_CANAL     = ['UNIDAD_CAPTURA', 'NOMBRE_UNIDAD_CAPTURA']
COLS_QUEJAS = [
    'QUEJAS_PENDIENTES', 'QUEJAS_RECIBIDAS', 'QUEJAS_FINALIZADAS',
    'QUEJAS_FINALIZADAS_N1', 'QUEJAS_FINALIZADAS_N2',
    'QUEJAS_EN_TRAMITE', 'QUEJAS_FAVOR_CONSUM_ACEP_ENT',
    'QUEJAS_FAVOR_CONSUM_NOACEP_ENT'
]

# Thresholds documentados en el enunciado
OUTLIER_EN_TRAMITE = 5_000

# Ventana de análisis de riesgo
ANIO_INICIO_RIESGO = 2021

In [9]:
# ¿N1 + N2 == FINALIZADAS?
check1 = ((df['QUEJAS_FINALIZADAS_N1'] + df['QUEJAS_FINALIZADAS_N2']).round(0) == df['QUEJAS_FINALIZADAS'].round(0)).mean()

# ¿ACEP + NOACEP == alguna de las columnas de finalizadas?
check2 = ((df['QUEJAS_FAVOR_CONSUM_ACEP_ENT'] + df['QUEJAS_FAVOR_CONSUM_NOACEP_ENT']).round(0) == df['QUEJAS_FINALIZADAS'].round(0)).mean()
check3 = ((df['QUEJAS_FAVOR_CONSUM_ACEP_ENT'] + df['QUEJAS_FAVOR_CONSUM_NOACEP_ENT']).round(0) == df['QUEJAS_FINALIZADAS_N2'].round(0)).mean()

print(f"N1 + N2 == FINALIZADAS: {check1:.2%}")
print(f"ACEP + NOACEP == FINALIZADAS: {check2:.2%}")
print(f"ACEP + NOACEP == N2: {check3:.2%}")

N1 + N2 == FINALIZADAS: 98.12%
ACEP + NOACEP == FINALIZADAS: 75.60%
ACEP + NOACEP == N2: 78.18%


## Hallazgos de Carga: Sección 0

| Hallazgo | Detalle | Acción |
|---|---|---|
| Trailing commas en nombre de columna | `QUEJAS_FAVOR_CONSUM_NOACEP_ENT,,,,,` | `str.rstrip(',')` en carga |
| Trailing commas en valores | `QUEJAS_FAVOR_CONSUM_NOACEP_ENT` tenía valores `'0,,,,,'`  cargada como string | Limpieza y conversión a float en carga |
| 16 filas completamente nulas | Nulos uniformes en todas las columnas | Eliminación en S1 |
| N1 y N2 son nivel de atención | N1 = primera línea, N2 = segunda línea (escaladas). N1+N2 = FINALIZADAS en 98.12% | Columnas se conservan con nombre original |
| ACEP + NOACEP no reconstruye FINALIZADAS | Coincidencia solo del 75.60% estas columnas representan un subconjunto, no el total | Investigar en S2 (EDA) |
| Columnas de resolución favor consumidor ausentes | El enunciado describe `QUEJAS_FINALIZAD_FAVOR_CONSUM` y `FAVOR_ENTIDAD`  no existen en el CSV | Variable objetivo se construirá desde `ACEP` y `NOACEP` decisión a justificar en S3 |